In [56]:
from pathlib import Path
import json

DATA_DIR = Path("/Users/calvin/all-hands/data/condenser-vibes")
CONDENSER_RUN = "field-type"

with (DATA_DIR / CONDENSER_RUN / "good.json").open("r") as f:
    good = json.load(f)
with (DATA_DIR / CONDENSER_RUN / "bad.json").open("r") as f:
    bad = json.load(f)

# Trim the first steps -- ill-formed
good = good[1:]
bad = bad[1:]

In [57]:
from typing import Any

def is_condensation(step: dict[str, Any]) -> bool:
    """
    Check if a step is a condensation step.
    """
    return "condensation" == step.get("action", None)


In [58]:
def render_step(step: dict[str, Any]) -> str:
    match step.get('action'):
        case "change_agent_state":
            return step.get("message")
        
        case "condensation":
            return step.get("message")
        
        case "finish":
            return "<FINISH>"
        
        case "message":
            return step.get("message")
        
        case "recall":
            return "<RECALL>"
        
        case "run":
            command = step.get("message")
            thought = step["tool_call_metadata"]["model_response"]["choices"][0]["message"]["content"]
            return f"<RUN {command}>\n{thought}"
        
        case "read":
            command = step.get("message")
            thought = step["tool_call_metadata"]["model_response"]["choices"][0]["message"]["content"]
            return f"<READ {command}>\n{thought}"
    
        case "edit":
            thought = step["tool_call_metadata"]["model_response"]["choices"][0]["message"]["content"]
            return f"<EDIT>\n{thought}"

        case "think":
            command = step.get("message")
            thought = step["tool_call_metadata"]["model_response"]["choices"][0]["message"]["content"]
            return f"<THINK {command}>\n{thought}"

    if "observation" in step:
        return "<OBSERVATION>"
    
    print(step)
    return step.get('message')

In [59]:
from rich.table import Table
from rich.markup import escape

table = Table(show_lines=True)
table.add_column("left")
table.add_column("right")

for index, (g, b) in enumerate(zip(good, bad)):
    table.add_row(escape(render_step(g)), escape(render_step(b)))

table

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ left                                                   ┃ right                                                  ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Agent state changed to awaiting_user_input             │ Agent state changed to awaiting_user_input             │
├────────────────────────────────────────────────────────┼────────────────────────────────────────────────────────┤
│ <OBSERVATION>                                          │ <OBSERVATION>                                          │
├────────────────────────────────────────────────────────┼────────────────────────────────────────────────────────┤
│ there's an input field for adding funds to your        │ there's an input field for adding funds to your        │
│ account on the billing page. it's a text input instead │ account on the billing page. it's a text input instead │
│ of a numerical input, please make it a number input    │ of a numerical input, please make it a number input    │
│ that only accepts ints                                 │ that only accepts ints                                 │
├────────────────────────────────────────────────────────┼────────────────────────────────────────────────────────┤
│ <RECALL>                                               │ <RECALL>                                               │
├────────────────────────────────────────────────────────┼────────────────────────────────────────────────────────┤
│ <OBSERVATION>                                          │ <OBSERVATION>                                          │
├────────────────────────────────────────────────────────┼────────────────────────────────────────────────────────┤
│ <OBSERVATION>                                          │ <OBSERVATION>                                          │
├────────────────────────────────────────────────────────┼────────────────────────────────────────────────────────┤
│ <RUN Running command: find frontend -type f -name      │ <RUN Running command: find frontend -type f -name      │
│ "*.tsx" -o -name "*.jsx" | grep -i bill>               │ "*.tsx" -o -name "*.jsx" | grep -i bill>               │
│ I'll help you change the input field for adding funds  │ I'll help you change the input field for adding funds  │
│ from a text input to a number input that only accepts  │ from a text input to a number input that only accepts  │
│ integers. Let's start by exploring the repository to   │ integers. Let's start by exploring the repository to   │
│ find the relevant files.                               │ find the relevant files.                               │
├────────────────────────────────────────────────────────┼────────────────────────────────────────────────────────┤
│ <OBSERVATION>                                          │ <OBSERVATION>                                          │
├────────────────────────────────────────────────────────┼────────────────────────────────────────────────────────┤
│ <READ Reading file:                                    │ <READ Reading file:                                    │
│ /workspace/OpenHands/frontend/src/routes/billing.tsx>  │ /workspace/OpenHands/frontend/src/routes/billing.tsx>  │
│ Great, I found the billing page file. Let's examine    │ Great, I found the billing page file. Let's examine    │
│ its content to locate the input field for adding       │ its content to locate the input field for adding       │
│ funds.                                                 │ funds.                                                 │
├────────────────────────────────────────────────────────┼────────────────────────────────────────────────────────┤
│ <OBSERVATION>                                          │ <OBSERVATION>                                          │
├───────────────────────────────────────────────────────

In [60]:
import ollama

embeddings = []
for step in bad:
    if "observation" in step:
        embeddings.append(None)
    else:
        embedding = ollama.embeddings(model="llama3.2", prompt=json.dumps(step))["embedding"]
        embeddings.append(embedding)

In [61]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

rows = []
for i in range(len(embeddings)):
    for j in range(len(embeddings)):
        if embeddings[i] is None or embeddings[j] is None:
            continue
        row = {
            "i": i,
            "j": j,
            "cosine_similarity": cosine_similarity([embeddings[i]], [embeddings[j]])[0][0],
        }
        rows.append(row)

df = pd.DataFrame(rows)

In [62]:
import altair as alt

alt.data_transformers.disable_max_rows()

alt.Chart(df).mark_rect(size=4).encode(
    x='i:O',
    y='j:O',
    color='cosine_similarity:Q',
    tooltip=['i', 'j', 'cosine_similarity'],
)

alt.Chart(...)

In [63]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from typing import List, Tuple, Optional
def convert_sparse_to_matrix(similarity_df: pd.DataFrame, default_value: float = 0.0) -> Tuple[np.ndarray, list]:
    """
    Convert sparse similarity representation to a full matrix,
    excluding rows and columns with no values.
    
    Args:
        similarity_df: DataFrame with columns [i, j, cosine_similarity]
        default_value: Value to use for missing pairs
        
    Returns:
        Tuple of:
        - Full similarity matrix as numpy array
        - List of original indices included in the filtered matrix
    """
    # Find indices with at least one similarity value
    active_indices: set[int] = set()
    
    for _, row in similarity_df.iterrows():
        i, j, sim = row['i'], row['j'], row['cosine_similarity']
        # Only consider non-None values
        if sim is not None:
            active_indices.add(i)
            active_indices.add(j)
    
    # Sort the active indices to maintain order
    active_indices = sorted(active_indices)
    
    # Create mapping from original indices to new indices
    index_map = {original: new for new, original in enumerate(active_indices)}
    
    # Create matrix with proper dimensions
    n = len(active_indices)
    matrix = np.ones((n, n)) * default_value
    
    # Fill in known values using the new indices
    for _, row in similarity_df.iterrows():
        i, j, sim = row['i'], row['j'], row['cosine_similarity']
        # Skip None values or indices not in our active set
        if sim is not None and i in active_indices and j in active_indices:
            new_i = index_map[i]
            new_j = index_map[j]
            matrix[new_i, new_j] = sim
            # Symmetric matrix - fill both sides of diagonal
            matrix[new_j, new_i] = sim
    
    # Set diagonal to 1.0 (each document is perfectly similar to itself)
    np.fill_diagonal(matrix, 1.0)
    
    return matrix, active_indices
def find_temporal_blocks(similarity_matrix: np.ndarray, 
                        threshold: float = 0.7,
                        min_block_size: int = 3) -> List[Tuple[int, int]]:
    """
    Find contiguous blocks along the diagonal of a similarity matrix.
    
    Args:
        similarity_matrix: Full similarity matrix
        threshold: Similarity threshold to consider as part of a block
        min_block_size: Minimum size of block to detect
        
    Returns:
        List of (start_idx, end_idx) tuples for each detected block
    """
    n = similarity_matrix.shape[0]
    blocks = []
    
    # Track the current block
    current_block_start = 0
    in_block = False
    
    for i in range(n):
        # Check if we're at the end of a block or still in one
        if i > 0:
            # See if the diagonal and its neighbors are above threshold
            # This looks at a small window around the current position
            window_size = min(3, i)
            window_indices = [(max(0, i-j), max(0, i-k)) 
                             for j in range(window_size) 
                             for k in range(window_size)]
            window_similarities = [similarity_matrix[idx] for idx in window_indices]
            window_similarity = np.mean(window_similarities)
            
            if window_similarity >= threshold:
                if not in_block:
                    # Start of a new block
                    current_block_start = i - 1
                    in_block = True
            else:
                if in_block:
                    # End of current block
                    block_size = i - current_block_start
                    if block_size >= min_block_size:
                        blocks.append((current_block_start, i - 1))
                    in_block = False
    
    # Handle the case where the last block extends to the end
    if in_block:
        block_size = n - current_block_start
        if block_size >= min_block_size:
            blocks.append((current_block_start, n - 1))
    
    return blocks

def detect_changepoints(similarity_matrix: np.ndarray, 
                      window_size: int = 5) -> List[int]:
    """
    Detect changepoints as drops in similarity along the diagonal.
    
    Args:
        similarity_matrix: Full similarity matrix
        window_size: Size of window to compare
        
    Returns:
        List of indices where changepoints occur
    """
    n = similarity_matrix.shape[0]
    
    # Calculate average similarity in sliding windows along diagonal
    diagonal_similarities = []
    for i in range(n):
        # Get window indices, respecting matrix boundaries
        start_idx = max(0, i-window_size)
        # Calculate mean similarity within the window
        window_sim = np.mean(similarity_matrix[start_idx:i+1, start_idx:i+1])
        diagonal_similarities.append(window_sim)
    
    # Convert to numpy array
    diagonal_similarities = np.array(diagonal_similarities)
    
    # Compute first differences to find drops
    diffs = np.diff(diagonal_similarities)
    
    # Smooth the differences to reduce noise
    smoothed_diffs = gaussian_filter(diffs, sigma=1.0)
    
    # Find significant negative drops (changepoints)
    threshold = np.mean(smoothed_diffs) - 1.5 * np.std(smoothed_diffs)
    changepoints = [i+1 for i in range(len(smoothed_diffs)) if smoothed_diffs[i] < threshold]
    
    return changepoints

def visualize_temporal_blocks(similarity_matrix: np.ndarray, 
                             blocks: List[Tuple[int, int]] = None,
                             changepoints: List[int] = None, 
                             timestamps: List[str] = None,
                             figsize: Tuple[int, int] = (12, 10)):
    """
    Visualize the similarity matrix with detected temporal blocks.
    
    Args:
        similarity_matrix: Full similarity matrix
        blocks: Detected block boundaries as (start, end) tuples
        changepoints: Detected changepoints
        timestamps: Optional list of timestamps for labeling
        figsize: Figure size
    """
    plt.figure(figsize=figsize)
    
    # Plot the similarity matrix
    plt.imshow(similarity_matrix, cmap='viridis', interpolation='nearest')
    plt.colorbar(label='Cosine Similarity')
    
    # Mark blocks if provided
    if blocks:
        for start, end in blocks:
            # Draw rectangle around block
            rect = plt.Rectangle((start-0.5, start-0.5), 
                               end-start+1, end-start+1, 
                               fill=False, edgecolor='red', linewidth=2)
            plt.gca().add_patch(rect)
            
            # Add block label
            plt.text(start + (end-start)//2, start-1.5, f"Block {blocks.index((start, end))+1}", 
                   color='red', fontsize=10, ha='center')
    
    # Mark changepoints if provided
    if changepoints:
        for cp in changepoints:
            plt.axhline(y=cp-0.5, color='yellow', linestyle='--', linewidth=1)
            plt.axvline(x=cp-0.5, color='yellow', linestyle='--', linewidth=1)
    
    # Add timestamps if provided
    if timestamps:
        # Only show a subset of timestamps if there are many
        if len(timestamps) > 20:
            step = len(timestamps) // 10
            tick_indices = np.arange(0, len(timestamps), step)
            plt.xticks(tick_indices, [timestamps[i] for i in tick_indices], rotation=45)
            plt.yticks(tick_indices, [timestamps[i] for i in tick_indices])
        else:
            plt.xticks(np.arange(len(timestamps)), timestamps, rotation=45)
            plt.yticks(np.arange(len(timestamps)), timestamps)
    
    plt.title("Similarity Matrix with Temporal Blocks")
    plt.tight_layout()
    plt.show()

def analyze_sparse_similarity(similarity_df: pd.DataFrame,
                             threshold: float = 0.7,
                             min_block_size: int = 3,
                             timestamps: Optional[List[str]] = None):
    """
    Analyze sparse similarity matrix to find temporal blocks.
    
    Args:
        similarity_df: DataFrame with columns [i, j, cosine_similarity]
        threshold: Similarity threshold for block detection
        min_block_size: Minimum block size
        timestamps: Optional list of timestamp strings
    """
    # Convert sparse representation to full matrix
    print("Converting sparse representation to matrix...")
    similarity_matrix = convert_sparse_to_matrix(similarity_df)
    
    # Find blocks
    print("Finding temporal blocks...")
    blocks = find_temporal_blocks(similarity_matrix, threshold, min_block_size)
    
    # Find changepoints
    print("Detecting changepoints...")
    changepoints = detect_changepoints(similarity_matrix)
    
    # Visualize
    print("Visualizing results...")
    visualize_temporal_blocks(
        similarity_matrix, 
        blocks, 
        changepoints, 
        timestamps
    )
    
    # Print block information
    print("\nDetected Temporal Blocks:")
    for i, (start, end) in enumerate(blocks):
        block = similarity_matrix[start:end+1, start:end+1]
        internal_similarity = np.mean(block)
        
        # External similarity (between this block and others)
        mask = np.ones(similarity_matrix.shape, dtype=bool)
        mask[start:end+1, start:end+1] = False
        external_similarity = np.mean(similarity_matrix[mask])
        
        print(f"\nBlock {i+1} (documents {start}-{end}):")
        print(f"  Size: {end-start+1} documents")
        print(f"  Internal similarity: {internal_similarity:.4f}")
        print(f"  External similarity: {external_similarity:.4f}")
        print(f"  Contrast: {(internal_similarity-external_similarity):.4f}")
        
        if timestamps and start < len(timestamps) and end < len(timestamps):
            print(f"  Time range: {timestamps[start]} to {timestamps[end]}")
    
    return similarity_matrix, blocks, changepoints

In [64]:
similarity_matrix, matrix_to_indices = convert_sparse_to_matrix(df)

In [80]:
blocks = find_temporal_blocks(similarity_matrix, threshold=0.8)

def block_to_document_indices(start, end):
    indices = matrix_to_indices[start:end+1]
    indices = [int(i) for i in indices]
    return indices

# grab original documents in blocks
block_docs = []
for start, end in blocks:
    block_docs.append(block_to_document_indices(start, end))

for block in block_docs:
    # grab original documents
    block = [bad[i].get('action') for i in block]
    
    print(block)

['change_agent_state', 'message', 'recall', 'run', 'read', 'run', 'read', 'read', 'read', 'edit', 'edit', 'edit', 'edit', 'edit', 'edit', 'run', 'run', 'run', 'run', 'run', 'run', 'run', 'condensation', 'read', 'read', 'read', 'run', 'read', 'edit', 'run', 'run', 'read', 'read', 'condensation']
['read', 'read', 'read', 'read', 'edit', 'run', 'think', 'read', 'read', 'condensation', 'read', 'read', 'read', 'read', 'read', 'read', 'edit', 'run', 'think', 'read', 'condensation', 'read', 'read', 'read', 'read', 'read', 'read', 'read', 'edit', 'edit', 'edit', 'condensation', 'run']


Condensation all over the blocks. Not much correlation with natural block boundaries is probably a good thing.

In [81]:
from typing import Iterable

changepoints = detect_changepoints(similarity_matrix, window_size=5)
# Convert changepoints to blocks of original indices

def changepoints_to_blocks(changepoints: list[int], entries: int) -> Iterable[tuple[int, int]]:
    cp = [0] + changepoints + [entries - 1]
    for start, end in zip(cp[:-1], cp[1:]):
        yield start, end

changepoint_docs = []
for start, end in changepoints_to_blocks(changepoints, len(similarity_matrix)):
    changepoint_docs.append(block_to_document_indices(start, end))

for block in changepoint_docs:
    # grab original documents
    block = [bad[i].get('action') for i in block]
    
    print(block)

['change_agent_state', 'message']
['message', 'recall']
['recall', 'run']
['run', 'read', 'run', 'read', 'read', 'read', 'edit', 'edit', 'edit', 'edit', 'edit', 'edit', 'run', 'run', 'run', 'run', 'run', 'run', 'run', 'condensation']
['condensation', 'read', 'read', 'read', 'run', 'read', 'edit', 'run', 'run', 'read', 'read', 'condensation', 'run', 'read', 'read', 'read', 'read', 'edit', 'run', 'think', 'read', 'read', 'condensation', 'read', 'read', 'read', 'read', 'read', 'read', 'edit', 'run', 'think', 'read', 'condensation', 'read', 'read', 'read', 'read', 'read', 'read', 'read', 'edit', 'edit', 'edit', 'condensation', 'run']


In [67]:
print(len(bad), len(good))

130 96
